In [1]:
import os
import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

C:\Users\Mansoor\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Mansoor\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


✓ Imports complete


In [2]:
df_train = pd.read_parquet("../data/processed/cleaned/train_clean.parquet")
df_val   = pd.read_parquet("../data/processed/cleaned/val_clean.parquet")
df_test  = pd.read_parquet("../data/processed/cleaned/test_clean.parquet")

# Restore timestamps
ts_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for df in [df_train, df_val, df_test]:
    for col in ts_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

print("✓ Cleaned splits loaded")
print(f"  Train : {df_train.shape}")
print(f"  Val   : {df_val.shape}")
print(f"  Test  : {df_test.shape}")

✓ Cleaned splits loaded
  Train : (67493, 40)
  Val   : (14473, 40)
  Test  : (14471, 40)


In [3]:
# All derived from order_purchase_timestamp — available at order time

def add_date_features(df):
    df = df.copy()
    ts = df["order_purchase_timestamp"]

    df["purchase_dow"]        = ts.dt.dayofweek          # 0=Mon, 6=Sun
    df["purchase_month"]      = ts.dt.month               # 1-12
    df["purchase_quarter"]    = ts.dt.quarter             # 1-4
    df["purchase_hour"]       = ts.dt.hour                # 0-23
    df["purchase_is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
    df["purchase_year"]       = ts.dt.year

    # Brazilian holidays (approximate fixed dates — major ones)
    # Used to flag orders placed near holidays
    HOLIDAYS = [
        "01-01", "04-21", "05-01", "09-07",
        "10-12", "11-02", "11-15", "12-25",
    ]
    month_day = ts.dt.strftime("%m-%d")
    df["purchase_near_holiday"] = month_day.isin(HOLIDAYS).astype(int)

    # Black Friday flag — 4th Friday of November
    is_nov = ts.dt.month == 11
    is_fri = ts.dt.dayofweek == 4
    is_late_nov = ts.dt.day >= 22
    df["purchase_black_friday"] = (is_nov & is_fri & is_late_nov).astype(int)

    return df

df_train = add_date_features(df_train)
df_val   = add_date_features(df_val)
df_test  = add_date_features(df_test)

date_features = [
    "purchase_dow", "purchase_month", "purchase_quarter",
    "purchase_hour", "purchase_is_weekend", "purchase_year",
    "purchase_near_holiday", "purchase_black_friday"
]

print("FEATURE GROUP 1 — Date Features\n")
print(f"  Features added: {date_features}")
print(f"\n  Value distributions (train):")
for col in date_features:
    vc = df_train[col].value_counts().sort_index()
    print(f"\n  [{col}]")
    for val, cnt in vc.items():
        pct = cnt / len(df_train) * 100
        print(f"    {val}: {cnt:,} ({pct:.1f}%)")

FEATURE GROUP 1 — Date Features

  Features added: ['purchase_dow', 'purchase_month', 'purchase_quarter', 'purchase_hour', 'purchase_is_weekend', 'purchase_year', 'purchase_near_holiday', 'purchase_black_friday']

  Value distributions (train):

  [purchase_dow]
    0: 10,949 (16.2%)
    1: 10,753 (15.9%)
    2: 10,626 (15.7%)
    3: 10,064 (14.9%)
    4: 9,514 (14.1%)
    5: 7,429 (11.0%)
    6: 8,158 (12.1%)

  [purchase_month]
    1: 5,465 (8.1%)
    2: 5,769 (8.5%)
    3: 6,683 (9.9%)
    4: 6,389 (9.5%)
    5: 7,318 (10.8%)
    6: 6,454 (9.6%)
    7: 6,949 (10.3%)
    8: 7,306 (10.8%)
    9: 2,913 (4.3%)
    10: 3,301 (4.9%)
    11: 5,082 (7.5%)
    12: 3,864 (5.7%)

  [purchase_quarter]
    1: 17,917 (26.5%)
    2: 20,161 (29.9%)
    3: 17,168 (25.4%)
    4: 12,247 (18.1%)

  [purchase_hour]
    0: 1,630 (2.4%)
    1: 789 (1.2%)
    2: 358 (0.5%)
    3: 184 (0.3%)
    4: 148 (0.2%)
    5: 128 (0.2%)
    6: 351 (0.5%)
    7: 841 (1.2%)
    8: 2,069 (3.1%)
    9: 3,265 (4.8%)
    1

In [4]:
# estimated_delivery_days = estimated_date - purchase_date
# This is the buffer Olist promises — likely the strongest predictor
# Available at order time (both dates are known when order is placed)

def add_delivery_window(df):
    df = df.copy()
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400
    return df

df_train = add_delivery_window(df_train)
df_val   = add_delivery_window(df_val)
df_test  = add_delivery_window(df_test)

print("FEATURE GROUP 2 — Estimated Delivery Window\n")
print(f"  Feature: estimated_delivery_days")
print(f"  (= estimated_delivery_date - purchase_timestamp, in days)")
print(f"\n  Train stats:")
print(df_train["estimated_delivery_days"].describe().round(3).to_string())
print(f"\n  Correlation with target (train): "
      f"{df_train['estimated_delivery_days'].corr(df_train['lead_time_variance']):.4f}")

FEATURE GROUP 2 — Estimated Delivery Window

  Feature: estimated_delivery_days
  (= estimated_delivery_date - purchase_timestamp, in days)

  Train stats:
count    67493.000
mean        23.756
std          8.679
min          2.026
25%         18.377
50%         23.250
75%         28.411
max        109.342

  Correlation with target (train): -0.5229


In [5]:
# Volumetric weight = (L × H × W) / 5000  (standard logistics formula)
# Volume = L × H × W (raw cubic cm)
# Density = actual_weight / volume

def add_product_features(df):
    df = df.copy()
    df["product_volume_cm3"]     = (
        df["product_length_cm"] *
        df["product_height_cm"] *
        df["product_width_cm"]
    )
    df["volumetric_weight"]      = df["product_volume_cm3"] / 5000
    df["freight_per_kg"]         = np.where(
        df["product_weight_g"] > 0,
        df["total_freight"] / (df["product_weight_g"] / 1000),
        0
    )
    df["freight_price_ratio"]    = np.where(
        df["total_price"] > 0,
        df["total_freight"] / df["total_price"],
        0
    )
    df["price_per_item"]         = df["total_price"] / df["item_count"]
    df["is_heavy_product"]       = (df["product_weight_g"] > 5000).astype(int)
    df["is_bulky_product"]       = (df["product_volume_cm3"] > 30000).astype(int)
    return df

df_train = add_product_features(df_train)
df_val   = add_product_features(df_val)
df_test  = add_product_features(df_test)

product_features = [
    "product_volume_cm3", "volumetric_weight", "freight_per_kg",
    "freight_price_ratio", "price_per_item", "is_heavy_product", "is_bulky_product"
]

print("FEATURE GROUP 3 — Product Features\n")
print(f"  {'Feature':<30} {'Mean':>10} {'Median':>10} {'Std':>10} {'Corr w/ target':>16}")
print("  " + "-" * 80)
for col in product_features:
    s = df_train[col]
    corr = s.corr(df_train["lead_time_variance"])
    print(f"  {col:<30} {s.mean():>10.3f} {s.median():>10.3f} "
          f"{s.std():>10.3f} {corr:>16.4f}")

FEATURE GROUP 3 — Product Features

  Feature                              Mean     Median        Std   Corr w/ target
  --------------------------------------------------------------------------------
  product_volume_cm3              14836.866   6358.000  22055.962           0.0096
  volumetric_weight                   2.967      1.272      4.411           0.0096
  freight_per_kg                     43.588     24.785     57.556          -0.0460
  freight_price_ratio                 0.303      0.224      0.275          -0.0335
  price_per_item                    120.157     79.000    144.367          -0.0086
  is_heavy_product                    0.124      0.000      0.330           0.0015
  is_bulky_product                    0.143      0.000      0.350           0.0045


In [6]:
# same_state flag, cross_region flag
# Brazilian regions grouped from states

REGION_MAP = {
    "sp": "southeast", "rj": "southeast", "mg": "southeast", "es": "southeast",
    "pr": "south",     "sc": "south",     "rs": "south",
    "mt": "centerwest","ms": "centerwest","go": "centerwest","df": "centerwest",
    "ba": "northeast", "se": "northeast", "al": "northeast", "pe": "northeast",
    "pb": "northeast", "rn": "northeast", "ce": "northeast", "pi": "northeast",
    "ma": "northeast",
    "pa": "north",     "am": "north",     "ac": "north",     "ro": "north",
    "rr": "north",     "ap": "north",     "to": "north",
}

def add_geo_features(df, region_map):
    df = df.copy()
    df["same_state"] = (
        df["customer_state"] == df["seller_state"]
    ).astype(int)

    df["customer_region"] = df["customer_state"].map(region_map).fillna("other")
    df["seller_region"]   = df["seller_state"].map(region_map).fillna("other")
    df["same_region"]     = (
        df["customer_region"] == df["seller_region"]
    ).astype(int)

    # Cross-region flag (broader than same_state)
    df["cross_region"] = (1 - df["same_region"]).astype(int)
    return df

df_train = add_geo_features(df_train, REGION_MAP)
df_val   = add_geo_features(df_val,   REGION_MAP)
df_test  = add_geo_features(df_test,  REGION_MAP)

geo_features = ["same_state", "same_region", "cross_region"]

print("FEATURE GROUP 4 — Geographic Features\n")
for col in geo_features:
    vc = df_train[col].value_counts()
    corr = df_train[col].corr(df_train["lead_time_variance"])
    print(f"  [{col}]  corr with target: {corr:.4f}")
    for val, cnt in vc.items():
        print(f"    {val}: {cnt:,} ({cnt/len(df_train)*100:.1f}%)")
    print()

print("  Customer region distribution (train):")
print(df_train["customer_region"].value_counts().to_string())
print("\n  Seller region distribution (train):")
print(df_train["seller_region"].value_counts().to_string())

FEATURE GROUP 4 — Geographic Features

  [same_state]  corr with target: 0.1196
    0: 43,263 (64.1%)
    1: 24,230 (35.9%)

  [same_region]  corr with target: 0.1095
    1: 41,950 (62.2%)
    0: 25,543 (37.8%)

  [cross_region]  corr with target: -0.1095
    0: 41,950 (62.2%)
    1: 25,543 (37.8%)

  Customer region distribution (train):
customer_region
southeast     46314
south          9644
northeast      6351
centerwest     3944
north          1240

  Seller region distribution (train):
seller_region
southeast     56319
south          9137
northeast      1042
centerwest      984
north            11


In [7]:
# Seller avg lead time variance and late rate — computed on TRAIN only
# This simulates "what do we know about this seller historically"
# Applied to val/test using the train-computed lookup

print("FEATURE GROUP 5 — Seller Historical Performance (fit on train only)\n")

seller_stats = df_train.groupby("primary_seller_id").agg(
    seller_avg_variance   = ("lead_time_variance", "mean"),
    seller_std_variance   = ("lead_time_variance", "std"),
    seller_order_count    = ("order_id", "count"),
    seller_late_rate      = ("lead_time_variance", lambda x: (x > 0).mean()),
).reset_index()

# Fill std=0 for sellers with only 1 order
seller_stats["seller_std_variance"] = seller_stats["seller_std_variance"].fillna(0)

print(f"  Unique sellers in train : {len(seller_stats):,}")
print(f"\n  Seller stats summary:")
print(seller_stats[["seller_avg_variance","seller_std_variance",
                      "seller_order_count","seller_late_rate"]].describe().round(3).to_string())

# Merge into all splits
# For unseen sellers in val/test → use global train means (cold start)
global_seller_avg  = df_train["lead_time_variance"].mean()
global_seller_std  = df_train["lead_time_variance"].std()
global_seller_late = (df_train["lead_time_variance"] > 0).mean()
global_seller_cnt  = seller_stats["seller_order_count"].median()

def add_seller_features(df, seller_stats,
                         global_avg, global_std,
                         global_late, global_cnt):
    df = df.merge(seller_stats, on="primary_seller_id", how="left")
    df["seller_avg_variance"]  = df["seller_avg_variance"].fillna(global_avg)
    df["seller_std_variance"]  = df["seller_std_variance"].fillna(global_std)
    df["seller_late_rate"]     = df["seller_late_rate"].fillna(global_late)
    df["seller_order_count"]   = df["seller_order_count"].fillna(global_cnt)
    return df

df_train = add_seller_features(df_train, seller_stats,
                                global_seller_avg, global_seller_std,
                                global_seller_late, global_seller_cnt)
df_val   = add_seller_features(df_val, seller_stats,
                                global_seller_avg, global_seller_std,
                                global_seller_late, global_seller_cnt)
df_test  = add_seller_features(df_test, seller_stats,
                                global_seller_avg, global_seller_std,
                                global_seller_late, global_seller_cnt)

seller_features = [
    "seller_avg_variance", "seller_std_variance",
    "seller_order_count", "seller_late_rate"
]

print(f"\n  Correlation with target (train):")
for col in seller_features:
    corr = df_train[col].corr(df_train["lead_time_variance"])
    print(f"    {col:<30} : {corr:.4f}")

# Check unseen seller coverage in val/test
val_unseen  = df_val["seller_avg_variance"].isnull().sum()
test_unseen = df_test["seller_avg_variance"].isnull().sum()
print(f"\n  Unseen sellers in val  : {val_unseen}")
print(f"  Unseen sellers in test : {test_unseen}")

FEATURE GROUP 5 — Seller Historical Performance (fit on train only)

  Unique sellers in train : 2,765

  Seller stats summary:
       seller_avg_variance  seller_std_variance  seller_order_count  seller_late_rate
count             2765.000             2765.000            2765.000          2765.000
mean               -11.513                6.555              24.410             0.084
std                  7.386                5.537              75.253             0.173
min                -65.322                0.000               1.000             0.000
25%                -14.735                2.360               2.000             0.000
50%                -11.189                6.688               5.000             0.000
75%                 -8.306                9.241              17.000             0.100
max                 50.233               75.055            1260.000             1.000

  Correlation with target (train):
    seller_avg_variance            : 0.3523
    seller_std_var

In [8]:
# Repeat customer flag — same customer_unique_id appearing multiple times in train

print("FEATURE GROUP 6 — Customer Features\n")

repeat_customers = (
    df_train.groupby("customer_unique_id")["order_id"]
    .count()
    .reset_index()
    .rename(columns={"order_id": "customer_order_count"})
)

repeat_customers["is_repeat_customer"] = (
    repeat_customers["customer_order_count"] > 1
).astype(int)

def add_customer_features(df, repeat_customers):
    df = df.merge(repeat_customers, on="customer_unique_id", how="left")
    df["customer_order_count"] = df["customer_order_count"].fillna(1)
    df["is_repeat_customer"]   = df["is_repeat_customer"].fillna(0).astype(int)
    return df

df_train = add_customer_features(df_train, repeat_customers)
df_val   = add_customer_features(df_val,   repeat_customers)
df_test  = add_customer_features(df_test,  repeat_customers)

print(f"  Repeat customers in train : "
      f"{df_train['is_repeat_customer'].sum():,} "
      f"({df_train['is_repeat_customer'].mean()*100:.2f}%)")
print(f"\n  Correlation with target:")
for col in ["customer_order_count", "is_repeat_customer"]:
    corr = df_train[col].corr(df_train["lead_time_variance"])
    print(f"    {col:<30} : {corr:.4f}")

FEATURE GROUP 6 — Customer Features

  Repeat customers in train : 3,033 (4.49%)

  Correlation with target:
    customer_order_count           : -0.0176
    is_repeat_customer             : -0.0192


In [9]:
# Target encoding for product_category_name_english
# Fit on train only — mean lead_time_variance per category

print("FEATURE GROUP 7 — Category Target Encoding (fit on train only)\n")

cat_target_enc = (
    df_train.groupby("product_category_name_english")["lead_time_variance"]
    .mean()
    .reset_index()
    .rename(columns={"lead_time_variance": "category_avg_variance"})
)

global_cat_avg = df_train["lead_time_variance"].mean()

def add_category_encoding(df, cat_target_enc, global_avg):
    df = df.merge(cat_target_enc, on="product_category_name_english", how="left")
    df["category_avg_variance"] = df["category_avg_variance"].fillna(global_avg)
    return df

df_train = add_category_encoding(df_train, cat_target_enc, global_cat_avg)
df_val   = add_category_encoding(df_val,   cat_target_enc, global_cat_avg)
df_test  = add_category_encoding(df_test,  cat_target_enc, global_cat_avg)

corr = df_train["category_avg_variance"].corr(df_train["lead_time_variance"])
print(f"  Categories encoded : {len(cat_target_enc)}")
print(f"  Global fallback    : {global_cat_avg:.4f}")
print(f"  Correlation with target : {corr:.4f}")
print(f"\n  Top 5 categories by avg variance (train):")
print(cat_target_enc.sort_values("category_avg_variance", ascending=False).head(5).to_string(index=False))
print(f"\n  Bottom 5 categories by avg variance (train):")
print(cat_target_enc.sort_values("category_avg_variance").head(5).to_string(index=False))

FEATURE GROUP 7 — Category Target Encoding (fit on train only)

  Categories encoded : 72
  Global fallback    : -11.2360
  Correlation with target : 0.0660

  Top 5 categories by avg variance (train):
    product_category_name_english  category_avg_variance
furniture_mattress_and_upholstery              -4.533517
                   home_comfort_2              -6.148234
            arts_and_craftmanship              -7.393595
                             food              -9.281157
                            audio              -9.324555

  Bottom 5 categories by avg variance (train):
product_category_name_english  category_avg_variance
        security_and_services             -20.316655
                   la_cuisine             -17.983421
               party_supplies             -14.643395
              fixed_telephony             -14.352351
            cds_dvds_musicals             -14.155604


In [10]:
# Target encode customer_state and seller_state
# Mean lead_time_variance per state — fit on train only

print("FEATURE GROUP 8 — State-Level Target Encoding (fit on train only)\n")

for state_col, new_col in [
    ("customer_state", "customer_state_avg_variance"),
    ("seller_state",   "seller_state_avg_variance"),
]:
    state_enc = (
        df_train.groupby(state_col)["lead_time_variance"]
        .mean()
        .reset_index()
        .rename(columns={"lead_time_variance": new_col})
    )
    global_avg = df_train["lead_time_variance"].mean()

    for df in [df_train, df_val, df_test]:
        df[new_col] = df[state_col].map(
            state_enc.set_index(state_col)[new_col]
        ).fillna(global_avg)

    corr = df_train[new_col].corr(df_train["lead_time_variance"])
    print(f"  {new_col}")
    print(f"    Unique states encoded : {len(state_enc)}")
    print(f"    Correlation w/ target : {corr:.4f}")
    print(f"    Top 3 states (worst variance):")
    for _, row in state_enc.sort_values(new_col, ascending=False).head(3).iterrows():
        print(f"      {row[state_col].upper()}: {row[new_col]:.3f}")
    print()

FEATURE GROUP 8 — State-Level Target Encoding (fit on train only)

  customer_state_avg_variance
    Unique states encoded : 27
    Correlation w/ target : 0.1303
    Top 3 states (worst variance):
      AL: -8.275
      MA: -8.354
      ES: -10.079

  seller_state_avg_variance
    Unique states encoded : 22
    Correlation w/ target : 0.1361
    Top 3 states (worst variance):
      AM: 9.347
      MA: -10.467
      SP: -10.476



In [11]:
# Define the exact feature columns to be used in modeling
# Exclude: IDs, raw timestamps, target, post-delivery cols,
#          raw string categoricals (replaced by encodings),
#          intermediate columns

FINAL_FEATURES = [
    # Date features
    "purchase_dow", "purchase_month", "purchase_quarter",
    "purchase_hour", "purchase_is_weekend", "purchase_year",
    "purchase_near_holiday", "purchase_black_friday",

    # Delivery window — expected strongest feature
    "estimated_delivery_days",

    # Order-level
    "item_count", "total_price", "total_freight", "avg_price",
    "unique_sellers", "unique_products",

    # Product features
    "product_weight_g", "product_length_cm", "product_height_cm",
    "product_width_cm", "product_volume_cm3", "volumetric_weight",
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "freight_per_kg", "freight_price_ratio", "price_per_item",
    "is_heavy_product", "is_bulky_product",

    # Payment features
    "total_payment_value", "payment_installments",
    "payment_methods_used",

    # Geographic features
    "same_state", "same_region", "cross_region",
    "customer_state_avg_variance", "seller_state_avg_variance",

    # Seller features
    "seller_avg_variance", "seller_std_variance",
    "seller_order_count", "seller_late_rate",

    # Customer features
    "customer_order_count", "is_repeat_customer",

    # Encoded categoricals
    "category_avg_variance",
]

TARGET = "lead_time_variance"

print(f"FINAL FEATURE SET\n")
print(f"  Total features : {len(FINAL_FEATURES)}")
print(f"\n  Feature list:")
for i, col in enumerate(FINAL_FEATURES, 1):
    available = "✓" if col in df_train.columns else "✗ MISSING"
    print(f"    {i:>2}. {col:<40} {available}")

# Check for any missing
missing = [c for c in FINAL_FEATURES if c not in df_train.columns]
print(f"\n  Missing features : {len(missing)}")
if missing:
    print(f"  {missing}")

FINAL FEATURE SET

  Total features : 44

  Feature list:
     1. purchase_dow                             ✓
     2. purchase_month                           ✓
     3. purchase_quarter                         ✓
     4. purchase_hour                            ✓
     5. purchase_is_weekend                      ✓
     6. purchase_year                            ✓
     7. purchase_near_holiday                    ✓
     8. purchase_black_friday                    ✓
     9. estimated_delivery_days                  ✓
    10. item_count                               ✓
    11. total_price                              ✓
    12. total_freight                            ✓
    13. avg_price                                ✓
    14. unique_sellers                           ✓
    15. unique_products                          ✓
    16. product_weight_g                         ✓
    17. product_length_cm                        ✓
    18. product_height_cm                        ✓
    19. product_width_cm

In [12]:
print("CORRELATION WITH TARGET — ALL FINAL FEATURES (train)\n")
print(f"  {'Feature':<40} {'Pearson Corr':>14}")
print("  " + "-" * 58)

corr_rows = []
for col in FINAL_FEATURES:
    if col in df_train.columns:
        corr = df_train[col].corr(df_train[TARGET])
        corr_rows.append((col, corr))

corr_rows_sorted = sorted(corr_rows, key=lambda x: abs(x[1]), reverse=True)
for col, corr in corr_rows_sorted:
    bar = "█" * int(abs(corr) * 50)
    direction = "▲" if corr > 0 else "▼"
    print(f"  {col:<40} {corr:>+.4f} {direction}  {bar}")

CORRELATION WITH TARGET — ALL FINAL FEATURES (train)

  Feature                                    Pearson Corr
  ----------------------------------------------------------
  estimated_delivery_days                  -0.5229 ▼  ██████████████████████████
  seller_avg_variance                      +0.3523 ▲  █████████████████
  seller_late_rate                         +0.2132 ▲  ██████████
  seller_state_avg_variance                +0.1361 ▲  ██████
  customer_state_avg_variance              +0.1303 ▲  ██████
  same_state                               +0.1196 ▲  █████
  same_region                              +0.1095 ▲  █████
  cross_region                             -0.1095 ▼  █████
  purchase_year                            +0.0772 ▲  ███
  category_avg_variance                    +0.0660 ▲  ███
  total_freight                            -0.0658 ▼  ███
  unique_sellers                           -0.0582 ▼  ██
  purchase_black_friday                    +0.0567 ▲  ██
  unique_products  

In [13]:
os.makedirs("../data/processed/featured", exist_ok=True)

# Save full dataframes (all cols — needed for inspection)
df_train.to_parquet("../data/processed/featured/train_featured.parquet", index=False)
df_val.to_parquet("../data/processed/featured/val_featured.parquet",     index=False)
df_test.to_parquet("../data/processed/featured/test_featured.parquet",   index=False)

# Save feature metadata
feature_meta = {
    "final_features"        : FINAL_FEATURES,
    "target"                : TARGET,
    "n_features"            : len(FINAL_FEATURES),
    "global_seller_avg"     : global_seller_avg,
    "global_seller_std"     : global_seller_std,
    "global_seller_late"    : global_seller_late,
    "global_cat_avg"        : global_cat_avg,
    "seller_stats_path"     : "data/processed/featured/seller_stats.parquet",
    "cat_encoding_path"     : "data/processed/featured/cat_target_enc.parquet",
    "region_map"            : REGION_MAP,
}

seller_stats.to_parquet("../data/processed/featured/seller_stats.parquet", index=False)
cat_target_enc.to_parquet("../data/processed/featured/cat_target_enc.parquet", index=False)

with open("../data/processed/featured/feature_metadata.json", "w") as f:
    json.dump(feature_meta, f, indent=2)

print("✓ Engineered splits saved:")
for fname in ["train_featured.parquet", "val_featured.parquet", "test_featured.parquet"]:
    path = f"../data/processed/featured/{fname}"
    size = os.path.getsize(path) / 1024 / 1024
    print(f"  {fname:<35} {size:.2f} MB  | {len(pd.read_parquet(path)):,} rows")

print(f"\n✓ Feature metadata saved")
print(f"✓ Seller stats saved")
print(f"✓ Category encoding saved")
print(f"\n  Final shapes:")
print(f"  Train : {df_train.shape}")
print(f"  Val   : {df_val.shape}")
print(f"  Test  : {df_test.shape}")

✓ Engineered splits saved:
  train_featured.parquet              14.74 MB  | 67,493 rows
  val_featured.parquet                3.52 MB  | 14,473 rows
  test_featured.parquet               3.50 MB  | 14,471 rows

✓ Feature metadata saved
✓ Seller stats saved
✓ Category encoding saved

  Final shapes:
  Train : (67493, 70)
  Val   : (14473, 70)
  Test  : (14471, 70)
